# C4 - Custom ReAct Agent with Configurable Personas
**Joey Saade | 202305993**

Use case: the Harborlight study advisor helps students and families match needs to published tutoring services. Tools: `lookup_business_fact` and `suggest_service`. Constraints: no invented prices, availability, bookings, or outcomes.

In [ ]:
%pip install -r requirements.txt

## Manual LangGraph ReAct loop
`agent.py` wires `think -> act -> think -> finish`. The model emits JSON with either an action and argument or an answer. The action node executes a local tool and records the observation. A four-call cap limits loops. No prebuilt agent executor is used.

In [ ]:
from agent import PERSONAS, INSTRUCTIONS, TOOL_DESCRIPTIONS, lookup_business_fact, suggest_service, ask
print(PERSONAS)
print(INSTRUCTIONS)
print(TOOL_DESCRIPTIONS)

## Persona and configuration design
The friendly advisor uses warm, plain language with low thinking and a 1024 token cap. The careful expert uses measured vocabulary with medium thinking and a 1536 token cap. The model configurations match the intended conversational styles.

In [ ]:
print(lookup_business_fact("programming"))
print(suggest_service("I need help with calculus"))

## Prompt experiments
I ran 24 live trials on `gemini-3.1-flash-lite`: two personas, four prompt variants, and three cases. The variants compare few-shot vs. zero-shot, a brief private decision check vs. no such check, and system vs. in-message instructions. The final answers and traces are in `evaluation_results.md` and `evaluation_live.jsonl`. The earlier run before the tool-first graph guard is preserved in `evaluation_baseline_results.md` and `evaluation_baseline.jsonl`. The keyword check is a task-specific screen, so I manually reviewed the price answers.

In [ ]:
import os
if os.getenv("GEMINI_API_KEY"):
    result = ask("Can you help with introductory Python?", "friendly_advisor")
    print(result["answer"])
    print(result["trace"])
else:
    print("Add GEMINI_API_KEY to .env, restart the kernel, and rerun.")

## Reflection

**Most helpful or natural persona:** The friendly advisor sounded warmer in discovery-call replies and averaged 38.5 words per answer. The careful expert was more direct at 33.3 words. For student-facing discovery, I prefer the friendly advisor; for concise factual guidance, I prefer the careful expert.

**Best prompt and configuration:** All eight combinations passed the three factual checks after revision, so this small experiment has no accuracy winner. I select the careful expert with few-shot system instructions for factual advising because it was concise, used tools on all three cases, and followed the worked example. Its medium thinking level and 1536 token cap support that measured style. Few-shot and brief-plan answers averaged about 31 words; zero-shot answers averaged 45.7.

**Reasoning and tools:** The baseline used a tool in 23/24 cases. The careful expert once answered an unpublished price question without a lookup. I added a graph guard that requests a lookup before any answer without an observation. The revised run used a tool in 24/24 cases, and each trace shows the action and observation.

**Effect of instructions:** The separate instructions defined JSON output, a worked example, tool use, and factual limits. They gave consistent action formats, but did not alone guarantee tool use in the baseline. The graph guard made that rule operational. No tested answer invented an hourly price.

**Biggest challenges:** Parsing model decisions reliably and working within the Gemini free-tier request limit. JSON mode, a four-step cap, a tool-first graph guard, and paced evaluation resolved these issues for the tested cases. The 24 trials cover only three questions and do not prove general reliability.